# Spambase 10/40/50 Resplit Setup

 10/40/50 for SVD/Train/Test

Outputs go to `spambase_10_40_50/`. The original `spambase_50_50/` is untouched.

## 1. Setup & Imports

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_STATE = 42
SOURCE_DIR = 'spambase_50_50'
OUTPUT_DIR = 'spambase_10_40_50'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')

Using device: cuda
GPU: NVIDIA RTX A6000
CUDA Version: 12.4


## 2. Load Existing 50/50 Artifacts

In [2]:
data = np.load(os.path.join(SOURCE_DIR, 'train_test_data.npz'))
X_train_old = data['X_train']
y_train_old = data['y_train']
X_test_old = data['X_test']
y_test_old = data['y_test']

prep = np.load(os.path.join(SOURCE_DIR, 'preprocessing.npz'))
weights = prep['weights']
bounds_min = prep['bounds_min']
bounds_max = prep['bounds_max']

with open(os.path.join(SOURCE_DIR, 'metadata.json'), 'r') as f:
    source_metadata = json.load(f)
feature_names = source_metadata['feature_names']

print(f'Loaded from {SOURCE_DIR}/:')
print(f'  X_train: {X_train_old.shape}, y_train: {y_train_old.shape}')
print(f'  X_test:  {X_test_old.shape}, y_test:  {y_test_old.shape}')
print(f'  weights: {weights.shape}, bounds_min: {bounds_min.shape}, bounds_max: {bounds_max.shape}')
print(f'  Class balance (train): 0={int((y_train_old == 0).sum())}, 1={int((y_train_old == 1).sum())}')
print(f'  Class balance (test):  0={int((y_test_old == 0).sum())}, 1={int((y_test_old == 1).sum())}')

Loaded from spambase_50_50/:
  X_train: (2300, 57), y_train: (2300,)
  X_test:  (2301, 57), y_test:  (2301,)
  weights: (57,), bounds_min: (57,), bounds_max: (57,)
  Class balance (train): 0=1394, 1=906
  Class balance (test):  0=1394, 1=907


## 3. Split

Split existing training set to 20% SVD and 80% training.

In [3]:
all_indices = np.arange(X_train_old.shape[0])
train_indices, svd_indices = train_test_split(
    all_indices,
    test_size=0.2,
    stratify=y_train_old,
    random_state=RANDOM_STATE,
)

X_train_new = X_train_old[train_indices]
y_train_new = y_train_old[train_indices]
X_svd = X_train_old[svd_indices]
y_svd = y_train_old[svd_indices]

print(f'New training partition (40% of dataset): {X_train_new.shape}')
print(f'  Class 0: {int((y_train_new == 0).sum())}, Class 1: {int((y_train_new == 1).sum())}')
print(f'SVD partition (10% of dataset):         {X_svd.shape}')
print(f'  Class 0: {int((y_svd == 0).sum())}, Class 1: {int((y_svd == 1).sum())}')

New training partition (40% of dataset): (1840, 57)
  Class 0: 1115, Class 1: 725
SVD partition (10% of dataset):         (460, 57)
  Class 0: 279, Class 1: 181


## 4. Verify Partitioning and Test-Set Preservation

The test set is reused verbatim from `spambase_50_50/`, so the Entry 5 noise scoring artifacts (`experiments/noise_scoring_outputs/20260324_203954/`) remain valid here. The training and SVD partitions must be disjoint and together cover the original `X_train` exactly.

In [4]:
X_test_new = X_test_old.copy()
y_test_new = y_test_old.copy()

assert np.array_equal(X_test_new, X_test_old), 'X_test must be byte-identical to spambase_50_50 X_test'
assert np.array_equal(y_test_new, y_test_old), 'y_test must be byte-identical to spambase_50_50 y_test'

train_set = set(train_indices.tolist())
svd_set = set(svd_indices.tolist())
assert train_set.isdisjoint(svd_set), 'training and SVD partitions overlap'
assert train_set | svd_set == set(range(X_train_old.shape[0])), 'partitions do not cover X_train_old'
assert X_train_new.shape[0] + X_svd.shape[0] == X_train_old.shape[0]

total = X_svd.shape[0] + X_train_new.shape[0] + X_test_new.shape[0]
print('Verifications passed:')
print(f'  X_test unchanged: True ({X_test_new.shape[0]} samples)')
print(f'  Partitions disjoint and complete: True')
print(f'  10/40/50 = {X_svd.shape[0]}/{X_train_new.shape[0]}/{X_test_new.shape[0]} = {total} total')
print(f'  Fractions: {X_svd.shape[0] / total:.4f} / {X_train_new.shape[0] / total:.4f} / {X_test_new.shape[0] / total:.4f}')

Verifications passed:
  X_test unchanged: True (2301 samples)
  Partitions disjoint and complete: True
  10/40/50 = 460/1840/2301 = 4601 total
  Fractions: 0.1000 / 0.3999 / 0.5001


## 5. Compute Clean-Basis SVD on the 460-Sample Partition


In [5]:
U_svd, S_svd, Vt_svd = np.linalg.svd(X_svd, full_matrices=False)

total_energy = (S_svd ** 2).sum()
print(f'U_svd:  {U_svd.shape}')
print(f'S_svd:  {S_svd.shape}')
print(f'Vt_svd: {Vt_svd.shape}')
print(f'Top-5 singular values: {S_svd[:5]}')
print(f'Energy in top-1:  {(S_svd[:1] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-5:  {(S_svd[:5] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-10: {(S_svd[:10] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-20: {(S_svd[:20] ** 2).sum() / total_energy:.2%}')

U_svd:  (460, 57)
S_svd:  (57,)
Vt_svd: (57, 57)
Top-5 singular values: [4.75969578 3.28441488 2.39255872 2.27923479 2.12940318]
Energy in top-1:  22.83%
Energy in top-5:  49.27%
Energy in top-10: 68.13%
Energy in top-20: 85.36%


## 6. Retrain MLP on the 40% Training Partition

In [6]:
class SpambaseNet(nn.Module):
    def __init__(self, D_in):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1),
        )

    def forward(self, x):
        return self.layer(x)


torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

D_in = X_train_new.shape[1]
model = SpambaseNet(D_in).to(device)

X_train_t = torch.FloatTensor(X_train_new).to(device)
y_train_t = torch.nn.functional.one_hot(
    torch.LongTensor(y_train_new), num_classes=2
).float().to(device)

X_test_t = torch.FloatTensor(X_test_new).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
N_EPOCHS = 100

print('Training...')
model.train()
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS} - Loss: {loss.item():.4f}')

model.eval()
print('Training complete.')

Training...
  Epoch  20/100 - Loss: 0.6647
  Epoch  40/100 - Loss: 0.6094
  Epoch  60/100 - Loss: 0.5073
  Epoch  80/100 - Loss: 0.3930
  Epoch 100/100 - Loss: 0.3062
Training complete.


## 7. Evaluate on Train and Test Sets

In [7]:
with torch.no_grad():
    train_preds = model(X_train_t).argmax(dim=1).cpu().numpy()
    test_preds = model(X_test_t).argmax(dim=1).cpu().numpy()

train_acc = (train_preds == y_train_new).mean()
test_acc = (test_preds == y_test_new).mean()

print(f'New 40% partition (1840 train samples):')
print(f'  Train Accuracy: {train_acc:.2%}')
print(f'  Test Accuracy:  {test_acc:.2%}')
print()
print(f'Reference (50% partition, Entry 5):')
print(f'  Train Accuracy: 91.09%')
print(f'  Test Accuracy:  89.96%')
print()
print('Test Set Classification Report:')
print(classification_report(y_test_new, test_preds, target_names=['Not Spam', 'Spam']))
print('Confusion Matrix:')
print(confusion_matrix(y_test_new, test_preds))

New 40% partition (1840 train samples):
  Train Accuracy: 90.98%
  Test Accuracy:  89.44%

Reference (50% partition, Entry 5):
  Train Accuracy: 91.09%
  Test Accuracy:  89.96%

Test Set Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.89      0.94      0.92      1394
        Spam       0.90      0.82      0.86       907

    accuracy                           0.89      2301
   macro avg       0.90      0.88      0.89      2301
weighted avg       0.89      0.89      0.89      2301

Confusion Matrix:
[[1313   81]
 [ 162  745]]


## 8. Save Artifacts

In [8]:
np.savez(
    os.path.join(OUTPUT_DIR, 'train_test_data.npz'),
    X_svd=X_svd, y_svd=y_svd,
    X_train=X_train_new, y_train=y_train_new,
    X_test=X_test_new, y_test=y_test_new,
    svd_indices=svd_indices, train_indices=train_indices,
)

np.savez(
    os.path.join(OUTPUT_DIR, 'preprocessing.npz'),
    weights=weights,
    bounds_min=bounds_min,
    bounds_max=bounds_max,
)

np.savez(
    os.path.join(OUTPUT_DIR, 'clean_svd_basis.npz'),
    U_svd=U_svd, S_svd=S_svd, Vt_svd=Vt_svd,
)

metadata = {
    'dataset': 'Spambase (UCI ID 94)',
    'split_ratio': '10/40/50 (SVD/Train/Test)',
    'random_state': RANDOM_STATE,
    'stratified': True,
    'n_features': D_in,
    'feature_names': feature_names,
    'n_svd': int(X_svd.shape[0]),
    'n_train': int(X_train_new.shape[0]),
    'n_test': int(X_test_new.shape[0]),
    'scaler': 'MinMaxScaler [0, 1] (inherited from spambase_50_50)',
    'derived_from': 'spambase_50_50/train_test_data.npz',
    'inner_split': '80/20 of existing X_train, stratified by y_train',
    'test_set_unchanged': True,
    'noise_scores_still_valid_at': 'experiments/noise_scoring_outputs/20260324_203954/',
}
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'SpambaseNet',
    'D_in': D_in,
    'hidden_layers': [64, 32],
    'n_classes': 2,
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'optimizer': 'Adam',
    'lr': 0.001,
    'epochs': N_EPOCHS,
    'loss': 'BCELoss',
    'trained_on': '40% partition (1840 samples) of spambase_10_40_50',
}, os.path.join(OUTPUT_DIR, 'spambase_mlp.pth'))

print(f'Artifacts saved to {OUTPUT_DIR}/:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(fpath)
    print(f'  {fname} ({size:,} bytes)')

Artifacts saved to spambase_10_40_50/:
  clean_svd_basis.npz (236,958 bytes)
  metadata.json (1,844 bytes)
  preprocessing.npz (2,140 bytes)
  spambase_mlp.pth (26,618 bytes)
  train_test_data.npz (2,155,262 bytes)
